In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

# spark = SparkSession.builder \
#     .appName("SparkExample") \
#     .master("local[*]") \
#     .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.memory', '8g') \
#     .config('spark.driver.memory', '8g') \
#     .getOrCreate()

## actualizar retiro telef 

In [ ]:

print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['TELEFONO'] = (
    df_list['TELEFONO']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 163965
df_list filas: 13150


In [ ]:

# telefonos = ["987863718", "991187143", "966700993", "997167060"]

# df_manual = pd.DataFrame(telefonos, columns=["TELEFONO"])


In [11]:
df_list = df_list.merge(
    df_long,
    on="TELEFONO",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 104


In [12]:

reemplazo = {
    'NUMERO_DOCUMENTO': 'col_01',
}
df_list = df_list.rename(columns=reemplazo)
df_list['col_02']='Retirar Telef'
df_list['col_03']=3
df_list=df_list[['col_01','col_02','col_03']]
df_list = df_list.drop_duplicates(
    subset=['col_01'],
    keep='first'
)
df_list.shape

(67, 3)

In [13]:
df_list.head()

,col_01,col_02,col_03
0,05709984,Retirar Telef,3
1,02895851,Retirar Telef,3
3,16014273,Retirar Telef,3
4,41980489,Retirar Telef,3
5,70314359,Retirar Telef,3


In [14]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 6867


In [15]:

df_list[['col_01', 'col_02', 'col_03']].to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

67

In [16]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.estado = b.col_02,
    a.cl_estado= b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 67


In [17]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes
SET 
    cl_estado=3
WHERE 
    cl_base = 'mayo 2026'
    and estado<> 'ACTIVO'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 1929


In [17]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 67


In [ ]:
# update_mysql_en_bloques(
#     df=df_list,
#     tabla="alfin_clientes",
#     periodo="mayo 2026",
#     col_llave_mysql="NUMERO_DOCUMENTO",
#     col_valor_mysql="estado",
#     col_llave_df="NUMERO_DOCUMENTO",
#     col_valor_df="retiro",
#     host=server_valentina,
#     user=user_valentina,
#     password=pwd_valentina,
#     database=db_valentina,
#     port=port_mysql,
#     batch_size=2000,
#     validar_sin_grabar=False
# )


Total registros a procesar: 6
Lote 0 - 6 actualizado | filas afectadas: 3
Proceso terminado. Total filas afectadas: 3


## actualizar retiro dni

In [18]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfin_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [19]:
reemplazo = {
    'dni_cliente': 'col_01',
}
df_dni = df_dni.rename(columns=reemplazo)

In [20]:
filename='RetiroDefinitivo_BlackList.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.rename(columns={'DNI': 'col_01'}, inplace=True)
df_list["col_01"] = (
    df_list["col_01"]
    .astype(str)
    .str.zfill(8)
)
df_list['col_03']=3
df_list['col_02']='Retirar Definitivo'
df_list=df_list[['col_01','col_02','col_03']]

df_list.head()

,col_01,col_02,col_03
0,00013948,Retirar Definitivo,3
1,00015508,Retirar Definitivo,3
2,00031376,Retirar Definitivo,3
3,00035800,Retirar Definitivo,3
4,00036260,Retirar Definitivo,3


In [21]:
df_list = df_list.merge(
    df_dni,
    on="col_01",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 0


In [14]:
df_list = df_list.drop_duplicates(
    subset=['col_01'],
    keep='first'
)
print(f"df_list filas: {df_list.shape[0]}")

df_list filas: 2


In [15]:
df_list.head()

,col_01,col_02,col_03
0,08739782,Retirar Definitivo,3
1,23690763,Retirar Definitivo,3


In [ ]:
df_list['col_03']=3
df_list['col_02']='Retirar definitivo'
df_list=df_list[['col_01','col_02','col_03']]

In [22]:
df_list['col_02']='Retirar Cali Opor'


In [22]:
filename='RetiroDeGestion_BlackList.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.rename(columns={'DNI': 'col_01'}, inplace=True)
df_list["col_01"] = (
    df_list["col_01"]
    .astype(str)
    .str.zfill(8)
)
df_list['col_03']=3
df_list['col_02']='Retirar Definitivo'
df_list=df_list[['col_01','col_02','col_03']]

df_list.head()

,col_01,col_02,col_03
0,03866035,Retirar Definitivo,3
1,03867988,Retirar Definitivo,3
2,03877469,Retirar Definitivo,3
3,03890737,Retirar Definitivo,3
4,23985035,Retirar Definitivo,3


In [23]:
df_list = df_list.merge(
    df_dni,
    on="col_01",
    how="inner"
)

df_list = df_list.drop_duplicates(
    subset=['col_01'],
    keep='first'
)
print(f"df_list filas: {df_list.shape[0]}")

df_list filas: 653


In [24]:
df_list.head()

,col_01,col_02,col_03
0,00029171,Retirar Definitivo,3
1,45266796,Retirar Definitivo,3
2,00229598,Retirar Definitivo,3
3,00240478,Retirar Definitivo,3
4,00371354,Retirar Definitivo,3


In [25]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 0


In [26]:

df_list[['col_01', 'col_02', 'col_03']].to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

653

In [23]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 1787


In [29]:
df_list.head()

,col_01,col_02,col_03
0,00029171,Retirar BlackList,3
1,45266796,Retirar BlackList,3
2,00229598,Retirar BlackList,3
3,00240478,Retirar BlackList,3
4,00371354,Retirar BlackList,3


In [28]:
df_list['col_02']='Retirar BlackList'
df_list.count()

col_01    653
col_02    653
col_03    653
dtype: int64

In [ ]:

reemplazo = {
    'dni_cliente': 'col_01',
    'retiro': 'col_02',
    'estado': 'col_03'
}
df_list = df_list.rename(columns=reemplazo)
df_list.head()

,col_01,col_02,col_03
0,05358759,Retirar BlackList,3
1,05377283,Retirar BlackList,3
2,09822790,Retirar BlackList,3
3,10459054,Retirar BlackList,3
4,10548156,Retirar BlackList,3


In [30]:

df_list[['col_01', 'col_02', 'col_03']].to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

653

In [17]:
df_list.head()

,col_01,col_02,col_03
0,22313090,Retirar Retiros por calificacion y oportunidad,3
1,10480794,Retirar Retiros por calificacion y oportunidad,3
2,32908380,Retirar Retiros por calificacion y oportunidad,3
3,10132661,Retirar Retiros por calificacion y oportunidad,3
4,18178794,Retirar Retiros por calificacion y oportunidad,3


In [19]:
print(
    df_list['col_01']
    .drop_duplicates()
    .tolist()
)

['20714264', '20902698', '20902790', '20976897', '20984392', '21002539', '22519705', '22530292', '22666436', '22759696', '22877147', '22880733', '22984404', '22994539', '22996414', '23001618', '23173921', '23254793', '23277023', '24468122', '24682712', '26697063', '26711897', '26713691', '26941590', '19323697', '19325492', '19329338', '19337234', '19409793', '19422299', '19422699', '19523495', '19527943', '19532970', '19559294', '19559342', '19560297', '19565034', '19572995', '19574719', '19693195', '19815345', '19847527', '19854392', '19862614', '19905252', '19918892', '21012899', '21094011', '21121648', '21133492', '21138395', '21264236', '21299643', '21427623', '21434302', '21444045', '21456695', '21480830', '23676691', '23835499', '23845415', '23864618', '23878432', '23894298', '24712094', '24872310', '24889591', '24991897', '25005499', '25300204', '19952315', '19954598', '19958290', '20000794', '20015896', '20020343', '20021691', '20022268', '20048803', '20073084', '20096899', '22

In [31]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.estado = b.col_02,
    a.cl_estado= b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 653


In [32]:
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFIN_BASE_VALENTINA", "SP actualizar BASE ALFIN")


SP actualizar BASE ALFIN | realizado | duración: 57.58 seg


In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfin_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.estado = b.col_02,
    a.cl_estado= b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

OperationalError: (pymysql.err.OperationalError) (1969, 'Query execution was interrupted (max_statement_time exceeded)')
[SQL: 
UPDATE crm_target.alfin_clientes a
INNER JOIN crm_target.tb_temporal b
    ON TRIM(CAST(a.NUMERO_DOCUMENTO AS CHAR)) = TRIM(CAST(b.col_01 AS CHAR))
SET 
    a.estado = b.col_02,
    a.cl_estado= b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# update_mysql_en_bloques(
#     df=df_list,
#     tabla="alfin_clientes",
#     periodo="mayo 2026",
#     col_llave_mysql="NUMERO_DOCUMENTO",
#     col_valor_mysql="estado",
#     col_llave_df="dni_cliente",
#     col_valor_df="retiro",
#     host=server_valentina,
#     user=user_valentina,
#     password=pwd_valentina,
#     database=db_valentina,
#     port=port_mysql,
#     batch_size=2000,
#     validar_sin_grabar=False
# )


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [ ]:
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfin_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
